# EMD Summary Figures — Mask Frequencies, Mean IMF Oscillations & Weighted HHT

Summary of the empirical mode decomposition (EMD) analysis produced by
[`emd_analysis_comprehensive.py`](emd_analysis_comprehensive.py). That script decomposes the
L4 MUA time series into 5 intrinsic mode functions (IMFs) for every **trial** (0–9) and
**stimulus direction** (`theta0`…`theta315`), i.e. **10 trials × 8 directions = 80 recordings**,
and stores per-recording results in `emd_analysis_output/emd_results_trial_*_angle_*.pkl`.

This notebook aggregates across all 80 recordings and produces three figures:

1. **Mask frequency per IMF** — box plot (median / IQR) with the individual recordings overlaid.
2. **Mean IMF oscillation** — trial-averaged IMF waveform per IMF with ± SEM shading
   (plus the mean instantaneous-amplitude envelope, which is not subject to phase cancellation).
3. **Mean IA-weighted Hilbert–Huang transform (HHT)** — mean power spectrum per IMF with ± SEM shading.
4. **Harmonic-relationship tests** — aggregated across all recordings to report the (lack of) harmonic
   structure between IMFs.

All figures are saved as vector **PDF** and 300-dpi **PNG** under `emd_analysis_output/figures/`.

> **Units note.** The `mask_freqs` stored in the pickles are already in **Hz** (they were returned by
> `emd.sift.iterated_mask_sift(..., sample_rate=fs)`); no `× fs` rescaling is applied here.


In [ ]:
import os
import glob
import pickle

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# ---- Publication-quality matplotlib defaults ------------------------------
mpl.rcParams.update({
    "figure.dpi": 120,          # on-screen
    "savefig.dpi": 300,         # exported raster
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,         # editable text in Illustrator/Inkscape
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "legend.fontsize": 9,
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

# ---- Per-IMF colour palette -----------------------------------------------
# The paper reserves red / blue / green / purple for other meanings, so those hue
# families are deliberately avoided here. Removing them leaves no colour-blind-safe
# *categorical* set (CVD compresses the remaining warm hues together), so we use a
# warm SEQUENTIAL ramp (YlOrBr, orange->brown) ordered by IMF: identity is carried
# by LIGHTNESS (robust to every CVD type) plus the axis labels / legend / peak
# position. One fixed colour per IMF, reused across every figure.
_ylorbr = mpl.colormaps["hsv"]
IMF_COLORS = [_ylorbr(x) for x in np.linspace(0.95, 0.05, 5)]  # IMF-1 darkest -> IMF-5 lightest
# Approximate physiological band each IMF falls in (for annotation only).
IMF_BANDS = ["gamma", "gamma", "alpha/beta", "theta", "delta"]

OUTPUT_DIR = "emd_analysis_output"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name):
    '''Save a figure as both PDF (vector) and PNG (300 dpi).'''
    for ext in ("pdf", "png"):
        path = os.path.join(FIG_DIR, f"{name}.{ext}")
        fig.savefig(path)
    print(f"saved: {os.path.join(FIG_DIR, name)}.{{pdf,png}}")

print("matplotlib", mpl.__version__)


## Load and stack all EMD results

We load every `emd_results_trial_*_angle_*.pkl` file and stack the quantities we need into
arrays with a leading *recording* axis (size 80):

| array | shape | meaning |
|-------|-------|---------|
| `mask_freqs`   | `(80, 5)`        | mask frequency (Hz) per IMF |
| `imf_waveforms`| `(80, 1249, 5)`  | IMF time series |
| `imf_ia`       | `(80, 1249, 5)`  | instantaneous amplitude (envelope) per IMF |
| `spec_weighted`| `(80, 128, 5)`   | IA-weighted HHT per IMF |
| `freq_centres` | `(128,)`         | HHT frequency-bin centres (Hz) |


In [ ]:
result_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "emd_results_trial_*_angle_*.pkl")))
print(f"Found {len(result_files)} result files")

mask_freqs, imf_waveforms, imf_ia, spec_weighted = [], [], [], []
trials, angles, harmonic_criteria_list = [], [], []
freq_centres = None
fs = None

for fp in result_files:
    with open(fp, "rb") as f:
        r = pickle.load(f)
    if "error" in r:
        print(f"  skipping {os.path.basename(fp)} (analysis error)")
        continue
    mask_freqs.append(np.asarray(r["mask_freqs"], dtype=float))
    imf_waveforms.append(np.asarray(r["imf_data"], dtype=float))
    imf_ia.append(np.asarray(r["instantaneous_amplitude"], dtype=float))
    spec_weighted.append(np.asarray(r["spec_weighted"], dtype=float))
    harmonic_criteria_list.append(r.get("harmonic_criteria"))
    trials.append(r["trial"])
    angles.append(r["angle"])
    if freq_centres is None:
        freq_centres = np.asarray(r["freq_centres"], dtype=float)
        fs = r["fs"]

mask_freqs   = np.stack(mask_freqs)      # (n_rec, 5)
imf_waveforms = np.stack(imf_waveforms)  # (n_rec, n_time, 5)
imf_ia       = np.stack(imf_ia)          # (n_rec, n_time, 5)
spec_weighted = np.stack(spec_weighted)  # (n_rec, n_freq, 5)

n_rec, n_time, n_imf = imf_waveforms.shape
time = np.arange(n_time) / fs            # seconds, relative to analysis window

print(f"n recordings : {n_rec}  ({len(set(trials))} trials x {len(set(angles))} directions)")
print(f"n IMFs       : {n_imf}")
print(f"sampling rate: {fs} Hz   |  window: {time[0]:.3f}-{time[-1]:.3f} s ({n_time} samples)")
print(f"HHT freq bins: {freq_centres.min():.2f}-{freq_centres.max():.2f} Hz ({freq_centres.size} bins)")


In [ ]:
# Per-IMF descriptive statistics for the mask frequencies (sanity table)
print(f"{'IMF':>4} {'band':>11} {'mean(Hz)':>9} {'SD':>7} {'SEM':>7} {'min':>7} {'max':>7}")
print("-" * 58)
for i in range(n_imf):
    x = mask_freqs[:, i]
    print(f"{i+1:>4} {IMF_BANDS[i]:>11} {x.mean():>9.2f} {x.std(ddof=1):>7.2f} "
          f"{x.std(ddof=1)/np.sqrt(n_rec):>7.3f} {x.min():>7.2f} {x.max():>7.2f}")
print("-" * 58)
print(f"pooled mean = {mask_freqs.mean():.3f} Hz   pooled SD (pop.) = {mask_freqs.std():.3f} Hz")
print("(compare with emd_analysis_output/analysis_summary.json: mean 25.051, std 25.990)")


## Figure 1 — Mask frequency per IMF (box plot)

Distribution of the mask (carrier) frequency for each of the 5 IMFs across all 80 trial ×
direction recordings. Boxes show the median and inter-quartile range (IQR), whiskers extend to
1.5 × IQR, and the dark dots are the 80 individual recordings (jittered) so the full spread is
visible. IMFs are ordered from highest (IMF-1) to lowest (IMF-5) carrier frequency, spanning the
gamma → delta range. The y-axis is logarithmic (the frequencies span ~2–73 Hz and each IMF's
spread is small), which shows that the mask frequency of each IMF is highly consistent across the
80 recordings.


In [ ]:
rng = np.random.default_rng(0)

positions = np.arange(n_imf)
data = [mask_freqs[:, i] for i in range(n_imf)]
medians = [float(np.median(d)) for d in data]

fig, ax = plt.subplots(figsize=(5, 4))

bp = ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True,
                showfliers=False, zorder=2,
                medianprops=dict(color="black", linewidth=1.6),
                boxprops=dict(linewidth=1.0, edgecolor="black"),
                whiskerprops=dict(color="black", linewidth=1.0),
                capprops=dict(color="black", linewidth=1.0))
for patch, c in zip(bp["boxes"], IMF_COLORS):
    patch.set_facecolor(c)
    patch.set_alpha(0.9)

# individual recordings, jittered
for i in range(n_imf):
    jitter = rng.uniform(-0.18, 0.18, size=n_rec)
    ax.scatter(positions[i] + jitter, mask_freqs[:, i],
               s=10, color="0.15", alpha=0.40, linewidths=0, zorder=3)

# log y-axis: mask frequencies span ~2-73 Hz, and each IMF's spread is small, so a
# log scale de-cramps the low-frequency IMFs and gives the boxes visible height.
from matplotlib.ticker import FixedLocator, NullLocator, ScalarFormatter
ax.set_yscale("log")
ax.set_ylim(1.2, 110)
ax.yaxis.set_major_locator(FixedLocator([2, 3, 5, 10, 20, 30, 50, 70, 100]))
ax.yaxis.set_minor_locator(NullLocator())
ax.yaxis.set_major_formatter(ScalarFormatter())

# median labels above each box
for i, med in enumerate(medians):
    ax.text(i, mask_freqs[:, i].max() * 1.07, f"{med:.1f} Hz",
            ha="center", va="bottom", fontsize=9)

ax.set_xticks(positions)
ax.set_xticklabels([f"IMF-{i+1}\n({b})" for i, b in enumerate(IMF_BANDS)])
ax.set_ylabel("Mask frequency (Hz)")
ax.set_xlabel("Intrinsic mode function")
ax.set_title("Mask frequency across trials\n& directions")
ax.margins(x=0.04)

fig.tight_layout()
savefig(fig, "fig1_suppl_mask_frequencies")
plt.show()


## Figure 2 — Mean IMF oscillation (trial-averaged) with ± SEM shading

Each IMF waveform is averaged across the 80 recordings at every time sample; the shaded band is
the ± SEM. Because IMFs are **not perfectly phase-aligned across recordings**, the trial average
captures the *stimulus-locked (evoked)* portion of each oscillation and attenuates the
non-phase-locked part (here the trial-averaged RMS is ~17–30 % of the single-recording RMS).
The average is therefore meaningful but smaller in amplitude than any single trace.

Note each IMF is plotted on its **own y-axis scale** (amplitudes differ by an order of magnitude
across IMFs) and in its assigned colour.


In [ ]:
mean_wave = imf_waveforms.mean(axis=0)                       # (n_time, 5)
sem_wave  = imf_waveforms.std(axis=0, ddof=1) / np.sqrt(n_rec)

fig, axes = plt.subplots(n_imf, 1, figsize=(7.0, 8.0), sharex=True)
for i, ax in enumerate(axes):
    m, s = mean_wave[:, i], sem_wave[:, i]
    ax.fill_between(time, m - s, m + s, color=IMF_COLORS[i], alpha=0.30, linewidth=0)
    ax.plot(time, m, color=IMF_COLORS[i], linewidth=1.3)
    ax.axhline(0, color="0.6", linewidth=0.6, zorder=0)
    ax.set_ylabel("Amplitude\n(a.u.)")
    ax.text(0.995, 0.92, f"IMF-{i+1}  ({IMF_BANDS[i]})",
            transform=ax.transAxes, ha="right", va="top", fontsize=10,
            color=IMF_COLORS[i], fontweight="bold")
    ax.margins(x=0)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("Mean IMF oscillation (trial-averaged, ± SEM)", y=0.995, fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.99))
savefig(fig, "fig1_suppl_mean_imf_oscillation")
plt.show()


### Figure 2b — Mean instantaneous-amplitude (envelope) with ± SEM shading

Complementary view that is **not** subject to phase cancellation: the mean instantaneous
amplitude (Hilbert envelope) of each IMF across the 80 recordings, with ± SEM shading. This shows
the amplitude time course of each oscillation directly.


In [ ]:
mean_ia = imf_ia.mean(axis=0)                       # (n_time, 5)
sem_ia  = imf_ia.std(axis=0, ddof=1) / np.sqrt(n_rec)

fig, ax = plt.subplots(figsize=(7.0, 4.2))
for i in range(n_imf):
    m, s = mean_ia[:, i], sem_ia[:, i]
    ax.fill_between(time, m - s, m + s, color=IMF_COLORS[i], alpha=0.20, linewidth=0)
    ax.plot(time, m, color=IMF_COLORS[i], linewidth=1.4,
            label=f"IMF-{i+1} ({IMF_BANDS[i]})")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Instantaneous amplitude (a.u.)")
ax.set_title("Mean IMF envelope across trials & directions (± SEM)")
ax.margins(x=0)
ax.legend(ncol=2, loc="upper right")
fig.tight_layout()
savefig(fig, "fig2b_mean_imf_envelope")
plt.show()


## Figure 3 — Mean IA-weighted Hilbert–Huang transform with ± SEM shading

Amplitude-weighted Hilbert–Huang power spectrum for each IMF, averaged across the 80 recordings,
with ± SEM shading. Each IMF occupies a distinct frequency band whose peak matches the
corresponding mask frequency in Figure 1 (IMF-5 ≈ 2 Hz delta up to IMF-1 broadband gamma), so the
decomposition cleanly separates the signal into non-overlapping oscillatory components.


In [ ]:
mean_spec = spec_weighted.mean(axis=0)                       # (n_freq, 5)
sem_spec  = spec_weighted.std(axis=0, ddof=1) / np.sqrt(n_rec)

fig, ax = plt.subplots(figsize=(5, 4))
for i in range(n_imf):
    m, s = mean_spec[:, i], sem_spec[:, i]
    ax.fill_between(freq_centres, m - s, m + s, color=IMF_COLORS[i], alpha=0.22, linewidth=0)
    ax.plot(freq_centres, m, color=IMF_COLORS[i], linewidth=1.5,
            label=f"IMF-{i+1} ({IMF_BANDS[i]})")
ax.set_xlim(0, 100)
ax.set_ylim(bottom=0)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("HHT power (a.u.)")
ax.set_title("Mean IA-weighted Hilbert–Huang transform (± SEM)")
ax.legend(ncol=2, loc="upper right")
fig.tight_layout()
savefig(fig, "fig2_suppl_weighted_hht")
plt.show()


## Harmonic-relationship tests — is any IMF a harmonic of another?

`emd_analysis_comprehensive.py` ran `emd.imftools.assess_harmonic_criteria(..., base_imf=4)` for
every recording, testing whether IMF-1…IMF-4 are **harmonics of IMF-5** (the slowest ≈ 2 Hz
component, taken as the potential fundamental). Following Fabus et al.
(*IEEE OJSP*, 2022), two IMFs are harmonically related only if **all three** criteria hold:

| # | Criterion | Column | Met when |
|---|-----------|--------|----------|
| 1 | **Integer frequency ratio** (IF ratio not different from nearest integer) | `Integer IF p-value` | p **>** 0.05 |
| 2 | **Consistent phase relationship** (phase–phase distance correlation) | `DistCorr p-value` | p **<** 0.05 |
| 3 | **af ratio < 1** (amplitude-ratio × frequency-ratio significantly below 1) | `af < 1 p-value` | p **<** 0.05 |

We aggregate these across all 80 recordings and classify each IMF pairing as *harmonic* only if all
three criteria are met simultaneously. A very low harmonic rate is evidence that the IMFs are
**independent oscillatory modes rather than harmonics** of one another.


In [ ]:
import pandas as pd

ALPHA = 0.05
base_imf_idx = 4  # base_imf used in emd_analysis_comprehensive.py (IMF-5, ~2 Hz)

# Stack the per-recording criteria into arrays of shape (n_rec, n_pairings)
crit = [pd.DataFrame(hc) for hc in harmonic_criteria_list if hc is not None]
n_pair = len(crit[0])                       # candidate harmonics: IMF-1..IMF-4 vs base
print(f"Recordings with harmonic criteria: {len(crit)}   |   candidate pairings each: {n_pair}")

fratio = np.vstack([d["InstFreq Ratio"].values      for d in crit])   # (n_rec, n_pair)
afval  = np.vstack([d["af Value"].values            for d in crit])
dcorr  = np.vstack([d["DistCorr"].values            for d in crit])
p_int  = np.vstack([d["Integer IF p-value"].values  for d in crit])
p_af   = np.vstack([d["af < 1 p-value"].values      for d in crit])
p_ph   = np.vstack([d["DistCorr p-value"].values    for d in crit])

met_int   = p_int > ALPHA          # integer frequency ratio
met_phase = p_ph  < ALPHA          # phase dependence
met_af    = p_af  < ALPHA          # af < 1
met_all   = met_int & met_phase & met_af

pair_labels = [f"IMF-{i+1} vs IMF-{base_imf_idx+1}" for i in range(n_pair)]
summary = pd.DataFrame({
    "freq ratio (mean)": fratio.mean(0),
    "nearest int":       np.round(fratio.mean(0)).astype(int),
    "integer-ratio %":   100 * met_int.mean(0),
    "af (mean)":         afval.mean(0),
    "af<1 %":            100 * met_af.mean(0),
    "phase-dep %":       100 * met_phase.mean(0),
    "HARMONIC % (all 3)":100 * met_all.mean(0),
}, index=pair_labels).round(2)

print(summary.to_string())

n_all = int(met_all.sum())
n_tot = met_all.size
print("\n" + "=" * 66)
print(f"Harmonic pairings meeting ALL three criteria: {n_all} / {n_tot} "
      f"({100*n_all/n_tot:.1f}%)")
print(f"Decisive criterion -> af < 1: met in {100*met_af.mean():.1f}% of pairings; "
      f"mean af = {afval.mean():.1f} (>> 1)")
print("Conclusion: no IMF is a harmonic of another -> the IMFs are")
print("independent oscillatory modes (no harmonic structure in the data).")
print("=" * 66)

summary.to_csv(os.path.join(OUTPUT_DIR, "harmonic_criteria_aggregate.csv"))
print(f"\nsaved: {os.path.join(OUTPUT_DIR, 'harmonic_criteria_aggregate.csv')}")


In [ ]:
# Save summary DataFrame as a standalone PDF table (Illustrator-ready).
# pdf.fonttype=42 is already set globally so all text is editable in Illustrator.

col_labels = list(summary.columns)
row_labels = list(summary.index)
cell_text  = summary.values.tolist()

n_rows = len(row_labels)
n_cols = len(col_labels)

col_widths = [0.18, 0.10, 0.16, 0.11, 0.09, 0.13, 0.20]   # tune per-column widths
row_height = 0.10    # inches per row
header_h   = 0.14   # header row taller for wrapped labels
fig_w = sum(col_widths) + 0.55   # left margin for row labels
fig_h = header_h + n_rows * row_height + 0.10

fig_tbl, ax_tbl = plt.subplots(figsize=(fig_w, fig_h))
ax_tbl.axis("off")

tbl = ax_tbl.table(
    cellText  = [[f"{v:.2f}" if isinstance(v, float) else str(v) for v in row]
                 for row in cell_text],
    rowLabels    = row_labels,
    colLabels    = col_labels,
    cellLoc      = "center",
    rowLoc       = "left",
    loc          = "center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.auto_set_column_width(list(range(n_cols)))

# Style header row
for j in range(n_cols):
    cell = tbl[0, j]
    cell.set_facecolor("#2e2e2e")
    cell.set_text_props(color="white", fontweight="bold")
    cell.set_height(header_h / fig_h)

# Style row-label column and alternate data row shading
for i in range(1, n_rows + 1):
    tbl[i, -1].set_text_props(fontweight="bold")   # row label
    bg = "#f5f5f5" if i % 2 == 0 else "white"
    for j in range(n_cols):
        tbl[i, j].set_facecolor(bg)

fig_tbl.tight_layout(pad=0.2)

pdf_path = os.path.join(FIG_DIR, "table_harmonic_criteria_summary.pdf")
fig_tbl.savefig(pdf_path, bbox_inches="tight")
print(f"saved: {pdf_path}")
plt.show()


### Figure 4 — Harmonic criteria across recordings

**Left:** the percentage of the 80 recordings in which each individual criterion — and all three
jointly ("harmonic") — is satisfied, per candidate IMF pairing (the value is printed above each bar,
so the `af < 1` and `all three` bars, which are 0 % everywhere, remain readable). **Right:** the af
ratio (mean ± SD across recordings) on a log axis; every pairing
sits far above the af = 1 threshold (dashed line), so criterion 3 fails throughout — the direct
evidence for the lack of harmonic structure.


In [ ]:
# Criterion colours sampled from the cividis colormap (no purple/green)
crit_names = ["integer-ratio", "af < 1", "phase-dep", "all three (harmonic)"]
crit_cols = mpl.colormaps["cividis"](np.linspace(0.08, 0.90, len(crit_names)))
af_color = crit_cols[1]   # colour of the "af < 1" criterion, reused in the right panel

fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.5, 1]})

# ---- Left: % of recordings meeting each criterion -------------------------
pct = np.vstack([100 * met_int.mean(0), 100 * met_af.mean(0),
                 100 * met_phase.mean(0), 100 * met_all.mean(0)])   # (4 crit, n_pair)
x = np.arange(n_pair)
w = 0.2
for k, name in enumerate(crit_names):
    bars = axL.bar(x + (k - 1.5) * w, pct[k], width=w, color=crit_cols[k],
                   edgecolor="black", linewidth=0.6, label=name)
    # value labels so the 0 % bars (af<1, all-three) remain readable
    axL.bar_label(bars, labels=[f"{v:.0f}" for v in pct[k]], padding=2,
                  fontsize=6.5, rotation=90)
axL.set_xticks(x)
axL.set_xticklabels(pair_labels, rotation=15)
axL.set_ylabel("Recordings meeting criterion (%)")
axL.set_ylim(0, 138)
axL.set_title("Harmonic criteria met (n = %d recordings)" % len(crit))
axL.legend(loc="upper center", ncol=2, fontsize=8, columnspacing=1.2)

# ---- Right: af ratio (mean +/- SD), log scale -----------------------------
af_mean, af_sd = afval.mean(0), afval.std(0, ddof=1)
axR.bar(x, af_mean, yerr=af_sd, width=0.6, color=af_color, edgecolor="black",
        linewidth=0.8, error_kw=dict(ecolor="black", elinewidth=1.0, capsize=4))
axR.axhline(1.0, color="black", linestyle="--", linewidth=1.2)
axR.text(n_pair - 0.5, 1.15, "af = 1 (harmonic threshold)", ha="right", va="bottom", fontsize=8)
axR.set_yscale("log")
axR.set_xticks(x)
axR.set_xticklabels(pair_labels, rotation=15)
axR.set_ylabel("af ratio (mean ± SD, log)")
axR.set_title("af ratio ≫ 1 for every pairing")

fig.tight_layout()
savefig(fig, "fig1_suppl_harmonic_criteria")
plt.show()


## Notes & data provenance

- **Source:** 80 pickles in `emd_analysis_output/` (10 trials × 8 directions), produced by
  `emd_analysis_comprehensive.py` from the L4 MUA-power signal
  (`dg_all_dirs_mua_power_layer_L4only_silence_inh`, `sample_rate = 500 Hz`).
- **Aggregation:** all statistics are computed across the 80 recordings; shading / error bars are
  **± SEM** (`SD / sqrt(80)`), matching the requested "SE" convention.
- **Mask-frequency units:** already in Hz in the pickles — no `× fs` rescaling.
- **Phase caveat (Figure 2):** trial-averaged waveforms reflect the phase-locked (evoked)
  component; the envelope view (Figure 2b) avoids phase cancellation.
- **Harmonic tests:** criteria from `emd.imftools.assess_harmonic_criteria` (base_imf = IMF-5),
  aggregated across recordings; the decisive `af < 1` criterion fails everywhere, so no IMF is a
  harmonic of another. Aggregated table saved to `emd_analysis_output/harmonic_criteria_aggregate.csv`.
- **Colour palette:** warm sequential ramp (YlOrBr) — the paper's reserved red/blue/green/purple are
  avoided; IMF identity is carried by lightness plus labels/legend.
- **Outputs:** vector PDF + 300-dpi PNG in `emd_analysis_output/figures/`.
